# Reading data from bronze volume only for venue & match meta data

In [0]:
from pyspark.sql.functions import col,concat_ws,explode,arrays_zip,expr,when,concat,lit,coalesce,last,map_entries,from_json, to_json,split,last,row_number,substring,right
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, StringType

In [0]:
iplData = spark.read.format('parquet') \
    .option('header','true') \
    .option('inferschema','true') \
    .load('/Volumes/ipl/bronze/ipldata')

In [0]:
info_event = iplData.withColumn('d',explode(arrays_zip(col('info.dates')))) \
.select(    col('info.event.name').alias('eventName'),
                                col('info.event.match_number').alias('matchNumber'),
                                col('info.overs').alias('overs'),
                                (substring(col('info.season'),-2,2).cast(IntegerType())+lit(2000)).alias('season'),
                                col('info.venue').alias('ground'),
                                col('info.team_type').alias('teamType'),
                                col('info.match_type').alias('matchType'),
                                col('info.gender').alias('gender'),
                                coalesce(col('info.city'),lit(split(col('info.venue'),' ')[0])).alias('city'),
                                col('d.dates').alias('dates'),
                            )
w_rn = Window.partitionBy('season').orderBy(col('dates'))
missingMatchNumber = info_event.withColumn('rn',row_number().over(w_rn))
cleansedEvenData = missingMatchNumber.select('eventName',
                                coalesce(col('matchNumber'),col('rn')).alias('matchNumber'),
                                col('overs'),
                                col('season'),
                                col('ground'),
                                col('teamType'),
                                col('matchType'),
                                col('gender'),
                                col('city'),
                                col('dates')
                                )

In [0]:
info_event.write.mode('overwrite') \
    .option('inferSchema','true') \
    .mode('overwrite') \
    .saveAsTable('ipl.silver.events')

In [0]:
explodedOfficials = iplData.withColumn('d',explode(arrays_zip(
    col('info.officials.match_referees'),
    col('info.officials.umpires'),
    col('info.officials.tv_umpires'),
    col('info.officials.reserve_umpires'),
    col('info.dates')
)))
selectedOfficials = explodedOfficials.select(
    col('info.event.name').alias('eventName'),
    col('info.event.match_number').alias('matchNumber'),
    (substring(col('info.season'),-2,2).cast(IntegerType())+lit(2000)).alias('season'),
    coalesce(col('d.match_referees'),lit('NA')).alias('matchReferees'),
    coalesce(col('d.umpires'),lit('NA')).alias('umpires'),
    coalesce(col('d.tv_umpires'),lit('NA')).alias('tvUmpires'),
    coalesce(col('d.reserve_umpires'),lit('NA')).alias('reserveUmpires'),
    col('d.dates').alias('dates')
)

semiCleansedOfficials = selectedOfficials.select(col('eventName')
                                                 ,coalesce(col('matchNumber'),row_number().over(w_rn)).alias('matchNumber')
                                                 ,col('season')
                                                 ,col('matchReferees')
                                                 ,col('umpires')
                                                 ,col('tvUmpires')
                                                 ,col('reserveUmpires')
                                                 )

unpivotOfficials = semiCleansedOfficials.selectExpr(
    "matchNumber",
    "season",
    "stack(4, 'matchReferees', matchReferees, 'tvUmpires', tvUmpires, 'umpires', umpires, 'reserveUmpires', reserveUmpires) as (officialType, officialName)"
)
display(selectedOfficials)

In [0]:
unpivotOfficials.dropna().write\
    .option('inferSchema','true') \
    .mode('overwrite') \
    .saveAsTable('ipl.silver.officials')

In [0]:
infoDates = iplData.withColumn('d',explode(arrays_zip(col('info.dates'))))
infoOutCome = infoDates.select(col('info.event.match_number').alias('matchNumber'),
                                                        col('info.season').alias('season'),
                                                        col('info.outcome.winner').alias('matchWinner'),
                                                        col('info.outcome.by.wickets').alias('wonByWickets'),
                                                        col('info.outcome.by.runs').alias('wonByRuns'),
                                                        col('info.toss.decision').alias('electedTo'),
                                                        col('info.toss.winner').alias('tossWinner'),
                                                        explode(col('info.player_of_match')).alias('playerOfTheMatch'),
                                                        col('d.dates').alias('dates')
)
# # ).withColumn('wonBy',when( col('info.outcome.by.runs').isNotNull(),'Runs').when(col('info.outcome.by.wickets').isNotNull(),'Wickets')).otherwise('No Results')
# display(infoOutCome)
infoOutComeCleansed = infoOutCome \
    .select(coalesce(col('matchNumber'),row_number().over(w_rn)).alias('matchNumber'),
            (substring(col('season'),-2,2).cast(IntegerType())+lit(2000)).alias('season'),
            coalesce(col('matchWinner'),lit('No Results')).alias('matchWinner'),
            coalesce(col('electedTo'),lit('No Toss')).alias('electedTo'),
            coalesce(col('tossWinner'),lit('No Toss')).alias('tossWinner'),
            coalesce(col('playerOfTheMatch'),lit('NA')).alias('playerOfTheMatch'),
            when(col("wonByRuns").isNotNull(), concat(lit("by "),col('wonByRuns'),lit(" Runs"))) \
            .when(col("wonByWickets").isNotNull(), concat(lit("by "),col('wonByWickets'),lit(" Wickets"))) \
            .otherwise(lit("No Results")).alias('wonBy'),
            )

In [0]:
infoOutComeCleansed.write.mode('overwrite').format('delta').saveAsTable('ipl.silver.eventOutcomes')

In [0]:
jsonDf = iplData.select(
    col("info.event.match_number").alias("matchNumber"),
    col("info.season").alias("season"),
    explode(col("info.dates")).alias('dates'),
    to_json(col("info.players")).alias("playersJson")
)

schema = "MAP<STRING, ARRAY<STRING>>"
mapDf = jsonDf.select("matchNumber", "season","dates",
                      from_json(col("playersJson"), schema).alias("playersMap"))



playersDf = (
    mapDf
    .select(coalesce(col("matchNumber"),row_number().over(w_rn)).alias("matchNumber")
            , (substring(col("season"),-2,2).cast(IntegerType())+lit(2000)).alias("season")
            , explode(col("playersMap")).alias("teamName", "players")
            )
    .withColumn("playerName", explode(col("players")))
    
)
finalPlayers = playersDf.select("matchNumber", "season", "teamName", "playerName")
display(finalPlayers)

In [0]:
finalPlayers.write.format('delta').mode('overwrite').saveAsTable('ipl.silver.players')

In [0]:
initialRegistryData = iplData.select(
    col('info.event.match_number').alias('matchNumber'),
    col('info.season').alias('season'),
    explode(col('info.dates')).alias('dates'),
    to_json(col('info.registry.people')).alias('registry')
    )
registrySchema = 'MAP<STRING, STRING>'
structRegistryData = initialRegistryData.select(coalesce(col('matchNumber'),row_number().over(w_rn)).alias('matchNumber')
                                                ,(substring(col('season'),-2,2).cast(IntegerType())+lit(2000)).alias('season')
                                                ,from_json(col('registry'),registrySchema).alias('registry')
                                                )
flattenedRegistryData = structRegistryData.select(col('matchNumber').alias('matchNumber')
                                                  ,col('season').alias('season')
                                                  ,explode(col('registry')).alias('playerName','playerID')
                                                  ).select(col('matchNumber'),col('season'),col('playerName'),col('playerID'))
# display(flattenedRegistryData)
flattenedRegistryData.write.format('delta').mode('overwrite').saveAsTable('ipl.silver.playerRegistry')
